# King2 Qwen2.5 3B — Colab System Info + 10-Minute Fine-Tuning

This notebook does two things:
1. Collects general system information about the Colab runtime.
2. Continues fine-tuning `RASHID778/king2-qwen2.5-3b` for ~10 minutes on a small instruction-following dataset.

Recommended runtime: **GPU T4** (Runtime → Change runtime type → GPU).

In [ ]:
# 📊 Collect system information
import sys
import platform
import subprocess
import json
from datetime import datetime

info = {
    "timestamp": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "gpu": "",
    "gpu_count": 0,
    "ram_gb": 0,
}

try:
    import torch
    info["torch_version"] = torch.__version__
    info["cuda_available"] = torch.cuda.is_available()
    info["gpu_count"] = torch.cuda.device_count()
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
except ImportError:
    info["torch_version"] = "Not installed"

try:
    import psutil
    info["ram_gb"] = round(psutil.virtual_memory().total / (1024 ** 3), 2)
except ImportError:
    pass

try:
    info["nvidia_smi"] = subprocess.check_output("nvidia-smi", shell=True, text=True)
except Exception:
    info["nvidia_smi"] = "nvidia-smi not available"

print(json.dumps(info, indent=2, default=str))

with open("/content/colab_system_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, default=str)

In [ ]:
# 📦 Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
# 🔧 Imports and setup
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import load_dataset
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EXISTING_ADAPTER = "RASHID778/king2-qwen2.5-3b"
OUTPUT_DIR = "/content/king2-qwen25-3b-continued"

MAX_STEPS = 60
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512

print("Transformers version:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 🧠 Load tokenizer and base model (4-bit for T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Base model loaded. GPU memory usage:")
print(subprocess.check_output("nvidia-smi", shell=True, text=True))

In [ ]:
# ➕ Merge existing adapter into base model, then add fresh LoRA
model = PeftModel.from_pretrained(base_model, EXISTING_ADAPTER)
print("Existing adapter loaded.")

model = model.merge_and_unload()
print("Existing adapter merged into base model.")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Memory optimizations
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [ ]:
# 📚 Load a small instruction-following dataset
def format_instruction(example):
    instruction = example.get("instruction", "")
    output = example.get("output", "")
    text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
    return {"text": text}

print("Loading dataset...")
dataset = load_dataset("tatsu-lab/alpaca", split="train[:200]")
dataset = dataset.map(format_instruction)
print(f"Dataset size: {len(dataset)} samples")

In [ ]:
# ⚙️ Training arguments (tuned for ~10 minutes on T4)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="steps",
    save_steps=30,
    warmup_steps=5,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

In [ ]:
# 🚀 Start training
import time

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f"\nTraining completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")

In [ ]:
# 💾 Save the new adapter and tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
print(f"Saved to: {OUTPUT_DIR}")
print("Files in output directory:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" - ", f)

In [ ]:
# 📋 Generate training summary
summary = {
    "base_model": BASE_MODEL,
    "existing_adapter": EXISTING_ADAPTER,
    "output_dir": OUTPUT_DIR,
    "max_steps": MAX_STEPS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "training_time_seconds": elapsed,
    "training_time_minutes": round(elapsed / 60, 2),
    "final_loss": trainer.state.log_history[-1].get("loss") if trainer.state.log_history else None,
}

print(json.dumps(summary, indent=2, default=str))

with open("/content/training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

In [ ]:
# 🧪 Quick inference test
prompt = "### Instruction:\nList three benefits of drinking water.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))